# Entrenamiento del clasificador Iris

En este notebook entreno un modelo propio de scikit-learn y lo guardo para usarlo desde mi API. Elegí Iris porque viene incluido en scikit-learn: puedo concentrarme en el ciclo completo sin sumar descargas ni limpieza externa.

## 1. Datos, variables y objetivo

Cada fila describe una flor. Uso como **features** el largo y el ancho del sépalo y del pétalo, en centímetros. La **target** es la especie: `setosa`, `versicolor` o `virginica`. Como el objetivo es una categoría, este problema es de **clasificación multiclase**, no de regresión.

In [1]:
from pathlib import Path
import json
import os

import joblib
import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = iris.data
y = iris.target
species_names = iris.target_names.tolist()
feature_names = iris.feature_names

print(f'Cantidad de filas: {len(X)}')
print(f'Variables de entrada: {feature_names}')
print(f'Clases objetivo: {species_names}')
print(f'Filas por especie: {dict(zip(species_names, np.bincount(y)))}')

Cantidad de filas: 150
Variables de entrada: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Clases objetivo: ['setosa', 'versicolor', 'virginica']
Filas por especie: {'setosa': np.int64(50), 'versicolor': np.int64(50), 'virginica': np.int64(50)}


## 2. Entrenamiento y evaluación

Separo el 25 % para evaluar con flores que el modelo no usó al aprender. Uso `stratify=y` para conservar la proporción de las tres especies y `random_state=42` para poder repetir la división. La pipeline aplica `StandardScaler` antes de una regresión logística. El escalado pone las variables en una escala comparable; la regresión logística es un clasificador multiclase y permite calcular probabilidades estimadas.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

accuracy = accuracy_score(y_test, predicciones)
macro_f1 = f1_score(y_test, predicciones, average='macro')
matriz = confusion_matrix(y_test, predicciones)
print(f'Accuracy: {accuracy:.3f}')
print(f'F1 macro: {macro_f1:.3f}')
print('Matriz de confusión (filas = especie real; columnas = predicción):')
print(matriz)
print('\nInforme por especie:')
print(classification_report(y_test, predicciones, target_names=species_names))

Accuracy: 0.921
F1 macro: 0.923
Matriz de confusión (filas = especie real; columnas = predicción):
[[12  0  0]
 [ 0 12  1]
 [ 0  2 11]]

Informe por especie:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        12
  versicolor       0.86      0.92      0.89        13
   virginica       0.92      0.85      0.88        13

    accuracy                           0.92        38
   macro avg       0.92      0.92      0.92        38
weighted avg       0.92      0.92      0.92        38



## 3. Interpretación

`Accuracy` mide la proporción total de aciertos. `F1 macro` calcula F1 por especie y promedia las tres por igual, así ninguna clase pesa más por su cantidad. La matriz de confusión muestra qué especies acerté y cuáles confundí. Iris tiene clases equilibradas, así que interpreto ambas métricas junto con la matriz. Es un conjunto pequeño: estos resultados no garantizan el mismo rendimiento en flores o mediciones de otro origen.

In [3]:
# Si ejecuto el notebook desde notebook/, subo un nivel; si lo ejecuto desde TP_Final/, uso esa carpeta.
raiz = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
carpeta_modelo = raiz / 'app' / 'modelo'
carpeta_frontend = raiz / 'app' / 'frontend'
carpeta_modelo.mkdir(parents=True, exist_ok=True)
carpeta_frontend.mkdir(parents=True, exist_ok=True)

ruta_modelo = carpeta_modelo / 'modelo.joblib'
joblib.dump(modelo, ruta_modelo)

# El frontend usa estos puntos para mostrar las medidas reales del dataset.
datos_grafico = [
    {
        'sepal_length_cm': float(fila[0]),
        'sepal_width_cm': float(fila[1]),
        'petal_length_cm': float(fila[2]),
        'petal_width_cm': float(fila[3]),
        'especie': species_names[int(etiqueta)],
    }
    for fila, etiqueta in zip(X, y)
]
with (carpeta_frontend / 'iris_data.js').open('w', encoding='utf-8') as archivo:
    archivo.write('window.IRIS_DATA = ' + json.dumps(datos_grafico, ensure_ascii=False) + ';\n')

peso_mb = os.path.getsize(ruta_modelo) / 1e6
print(f'Modelo guardado en: {ruta_modelo}')
print(f'Versión de scikit-learn: {sklearn.__version__}')
print(f'Tamaño del archivo: {peso_mb:.4f} MB')
print(f'Puntos preparados para el gráfico web: {len(datos_grafico)}')

Modelo guardado en: C:\Users\grise\Proyectos\Tercer Año\Seminario de Actualizacion\Seminario2026\Clase 13\TP_Final\app\modelo\modelo.joblib
Versión de scikit-learn: 1.8.0
Tamaño del archivo: 0.0016 MB
Puntos preparados para el gráfico web: 150


## 4. Cierre

Guardé la pipeline completa para que la API aplique el mismo escalado del entrenamiento. La API recibirá las cuatro medidas, las validará con Pydantic y consultará el archivo `modelo.joblib`. El frontend mostrará la especie y las probabilidades estimadas. El gráfico de dispersión permite explorar cómo se relacionan las dos medidas del pétalo con las etiquetas; no es un gráfico de regresión porque mi objetivo es una categoría.